In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from timeit import default_timer as timer

from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, Rescaling, AvgPool2D, BatchNormalization, Reshape # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler # type: ignore

dataset = "/mnt/d/datasets/spanish_traffic/Classification/samples"

labelfile = pd.read_csv("/mnt/d/datasets/spanish_traffic/Classification/" + "gt_spanish_dataset.csv")

In [39]:
labelfile.head()

,image,width,height,class_id,class_name
0,Image00001.jpg,174,174,1,R-1
1,Image00002.jpg,240,240,1,R-1
2,Image00003.jpg,320,320,1,R-1
3,Image00004.jpg,120,120,1,R-1
4,Image00005.jpg,86,86,1,R-1


In [69]:
import tensorflow as tf
from tensorflow.keras import layers, models  # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint # type: ignore

img_size = (128, 128)
batch_size = 32
validation_split = 0.1
seed = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset,
    validation_split=validation_split,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

# Load validation / test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset,
    validation_split=validation_split,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

Found 1478 files belonging to 99 classes.
Using 1331 files for training.
Found 1478 files belonging to 99 classes.
Using 147 files for validation.


In [ ]:
num_classes = 99 

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = models.Sequential([
    layers.InputLayer(input_shape=(128, 128, 3)),
    layers.Rescaling(1./255),
    data_augmentation,        
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint('model.keras', monitor='val_accuracy', save_best_only=True)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=100,
    callbacks=[checkpoint]
)

best_model = tf.keras.models.load_model('model.keras')
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.1580 - loss: 3.9934 - val_accuracy: 0.3673 - val_loss: 3.0001
Epoch 2/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.4030 - loss: 2.7718 - val_accuracy: 0.4490 - val_loss: 2.4181
Epoch 3/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.4758 - loss: 2.2520 - val_accuracy: 0.4898 - val_loss: 2.1615
Epoch 4/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.5229 - loss: 2.0067 - val_accuracy: 0.5578 - val_loss: 1.9669
Epoch 5/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.5732 - loss: 1.7624 - val_accuracy: 0.6054 - val_loss: 1.7678
Epoch 6/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6286 - loss: 1.5526 - val_accuracy: 0.5986 - val_loss: 1.6559
Epoch 7/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.6547 - loss: 1.3703 - val_accuracy: 0.6395 - val_loss: 1.5159
Epoch 8/100
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.6654 - loss: 1.2617 - val_accuracy: 0.